# Stage 01 - AOI visualization


In [ ]:
#@title Display AOI
%pip install -q "geopandas>=1.0,<2" "folium>=0.17,<1"
import json

import folium
import geopandas as gpd
import pandas as pd
from branca.element import Element
from IPython.display import Markdown, display
from shapely.geometry import Polygon

ANALYSIS_CRS = "EPSG:32633"
EXCHANGE_CRS = "EPSG:4326"

VERTICES = [
    ("A1", 375142.89553203, 730553.78023993, 927),
    ("A2", 381315.67865558, 726934.57048810, 952),
    ("A3", 386655.47274849, 730036.88441500, 933),
    ("A4", 381124.21918453, 733184.44527635, 918),
    ("A5", 382254.47034824, 735241.34560231, 914),
    ("A6", 380256.00132039, 736073.12628858, 912),
    ("A7", 376031.75713711, 732703.05038135, 925),
]

polygon = Polygon([(x, y) for _, x, y, _ in VERTICES])
aoi_utm = gpd.GeoDataFrame(
    [{"aoi_id": "Parcel A"}],
    geometry=[polygon],
    crs=ANALYSIS_CRS,
)
area_ha = float(aoi_utm.geometry.area.iloc[0] / 10_000)
aoi_utm["area_ha"] = round(area_ha, 2)
aoi_wgs84 = aoi_utm.to_crs(EXCHANGE_CRS)

centroid_utm = aoi_utm.geometry.centroid.iloc[0]
centroid = gpd.GeoSeries([centroid_utm], crs=ANALYSIS_CRS).to_crs(EXCHANGE_CRS).iloc[0]
aoi_map = folium.Map(
    location=[centroid.y, centroid.x],
    zoom_start=12,
    tiles=None,
    control_scale=True,
)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satellite",
    show=True,
).add_to(aoi_map)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri Light Gray Canvas",
    name="Light map",
    show=False,
).add_to(aoi_map)
folium.TileLayer("OpenStreetMap", name="Street map", show=False).add_to(aoi_map)
boundary_layer = folium.GeoJson(
    data=json.loads(aoi_wgs84.to_json()),
    name="Parcel A boundary",
    style_function=lambda _: {
        "color": "#24573a",
        "weight": 4,
        "fillColor": "#80b918",
        "fillOpacity": 0.30,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["aoi_id", "area_ha"],
        aliases=["AOI", "Area (ha)"],
    ),
).add_to(aoi_map)

vertex_gdf = gpd.GeoDataFrame(
    [
        {"Point": point, "X": x, "Y": y, "Elevation_m": elevation}
        for point, x, y, elevation in VERTICES
    ],
    geometry=gpd.points_from_xy(
        [x for _, x, _, _ in VERTICES],
        [y for _, _, y, _ in VERTICES],
    ),
    crs=ANALYSIS_CRS,
).to_crs(EXCHANGE_CRS)
for row in vertex_gdf.itertuples():
    folium.CircleMarker(
        [row.geometry.y, row.geometry.x],
        radius=5,
        color="#9d2a2a",
        fill=True,
        fill_color="white",
        fill_opacity=1,
        tooltip=f"{row.Point}: X={row.X:.2f}, Y={row.Y:.2f}",
        popup=f"<b>{row.Point}</b><br>X: {row.X:.2f} m<br>Y: {row.Y:.2f} m",
    ).add_to(aoi_map)
    folium.Marker(
        [row.geometry.y, row.geometry.x],
        icon=folium.DivIcon(
            html=f'<div style="color:white;font-weight:bold;text-shadow:0 0 3px black;">{row.Point}</div>'
        ),
    ).add_to(aoi_map)

folium.LayerControl(collapsed=False).add_to(aoi_map)
minx, miny, maxx, maxy = aoi_wgs84.total_bounds
aoi_map.fit_bounds([[miny, minx], [maxy, maxx]])

area_panel = Element(f'''
<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;
padding:8px 12px;border:2px solid #24573a;border-radius:6px;font-weight:bold;">
AOI area: {area_ha:,.2f} ha
</div>
''')
aoi_map.get_root().html.add_child(area_panel)

point_table = pd.DataFrame(
    VERTICES,
    columns=["Point", "X (m)", "Y (m)", "Elevation (m)"],
)
display(Markdown(f"## AOI area: {area_ha:,.2f} ha"))
display(point_table.style.format({"X (m)": "{:.2f}", "Y (m)": "{:.2f}"}).hide(axis="index"))
display(aoi_map)
